# Precomputed functional claims


**What this precomputes (instruction 68.02).** For genes with no EC annotation in Toxoplasma and Plasmodium: the probability that each EC major class applies, from evidence that cannot restate how EC numbers are assigned, each with the held-out grade of its class.

**Held out as evidence:** every column of the function and domain categories, and the EC, InterPro, Pfam and EC-by-orthology families -- EC assignments are commonly made from domains and orthologs, so those inputs would partly restate the answer.

Built 2026-11-08.

In [1]:
import json, os, re, numpy as np, pandas as pd
from parascape import datasets as ds, harness as H, information_categories as IC, organisms as O
from parascape.question_registry import Question
EC = re.compile(r'\b([1-7])\.(?:\d+|-)\.(?:\d+|-)\.(?:n?\d+|-)')
NAMES = {'1': 'oxidoreductase', '2': 'transferase', '3': 'hydrolase', '4': 'lyase', '5': 'isomerase', '6': 'ligase', '7': 'translocase'}
def majors(v):
    return sorted({m.group(1) for m in EC.finditer(str(v))}) if isinstance(v, str) else []

## 1 · Per organism and class: grade, then deploy

In [2]:
claims, grades = [], []
for org, table in ((O.TOXOPLASMA, 'nodes.parquet'), (O.FALCIPARUM, 'pf_nodes.parquet')):
    nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data', table)).set_index('gene_id')
    numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
    fam = lambda c, org=org: (ds.provenance(c, org).key if ds.provenance(c, org) else 'unregistered')
    families = pd.Series({c: fam(c) for c in numeric})
    def blocked(c):
        try:
            cats = set(IC.categories_of(c, org))
        except Exception:
            cats = set()
        f = families.get(c, '')
        return bool(cats & {'function', 'domain'}) or any(k in f.lower() for k in ('ec', 'interpro', 'pfam', 'domain'))
    keep = [c for c in numeric if not blocked(c)]
    groups = nodes['orthogroup'].astype(object) if 'orthogroup' in nodes else pd.Series(nodes.index, index=nodes.index)
    groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in nodes.index}))
    sets = nodes['ec_number'].map(majors)
    annotated = sets[sets.map(len) > 0]
    unannotated = nodes.index[sets.map(len) == 0]
    ec_family = fam('ec_number') if 'ec_number' in families else 'ec'
    print(org, len(annotated), 'annotated;', len(unannotated), 'without EC;', len(keep), 'evidence columns of', len(numeric))
    for k in sorted({m for s in annotated for m in s}):
        labels = annotated.map(lambda s, k=k: k in s)
        if labels.sum() < 50 or (~labels).sum() < 50:
            continue
        q = Question(id=f'function_ec{k}_{org}', text=f'Is this {org} enzyme a {NAMES[k]} (EC {k})?', answer_type='boolean',
                     question_class='B', label_source=f'{table}:ec_number', label_family=ec_family,
                     evidence_forbidden=(ec_family,), label_tier='observed-curated', label_category='function', status='scored')
        feats = nodes.loc[labels.index, keep].astype('float64')
        feats = H.allowed_features(q, feats, families)
        r = H.run_question(q, labels=labels.astype(bool), features=feats, families=families, groups=groups.reindex(labels.index),
                           strategies=[H.Strategy.of('boosted trees', 'hist_gbt', max_iter=150)], seed=0, organism=org,
                           importance_repeats=1, log_path=H.HOLDOUT_LOG)
        ft = r.frozen_test or {}
        grades.append(dict(organism=org, ec_class=k, name=NAMES[k], annotated=len(labels), positives=int(labels.sum()),
                           frozen_auc=ft.get('roc_auc'), average_precision=ft.get('average_precision'), share=float(labels.mean()),
                           precision_lower90=ft.get('precision_lower90'), may_fill=bool(r.may_fill),
                           top_evidence=', '.join(r.importance.head(3)['family'].tolist()) if len(r.importance) else ''))
        dep = H.deploy(r, nodes.loc[unannotated, feats.columns].astype('float64'))
        claims.append(pd.DataFrame({'organism': org, 'gene_id': unannotated.astype(str), 'ec_class': k, 'name': NAMES[k],
                                    'probability': dep.probability.round(4).values, 'calibrated': dep.calibrated.round(4).values,
                                    'called': dep.called.values, 'fills': dep.fills.values,
                                    'class_frozen_auc': round(ft.get('roc_auc', np.nan), 3)}))
G = pd.DataFrame(grades); C = pd.concat(claims, ignore_index=True)
G.to_csv(os.path.join(OUT, 'grades.tsv'), sep='\t', index=False)
G.round(3)

Tg 1313 annotated; 6827 without EC; 380 evidence columns of 411
Pf 1220 annotated; 4500 without EC; 147 evidence columns of 151


,organism,ec_class,name,annotated,positives,frozen_auc,average_precision,share,precision_lower90,may_fill,top_evidence
0,Tg,1,oxidoreductase,1313,166,0.731,0.356,0.126,NaN,False,"gse206344, iron_depletion_proteome, unregistered"
1,Tg,2,transferase,1313,547,0.725,0.639,0.417,0.681,False,"foldseek_struct, crispr_serum_restriction, local_af3"
2,Tg,3,hydrolase,1313,420,0.667,0.575,0.32,NaN,False,"unregistered, foldseek_struct, gse19092"
3,Tg,4,lyase,1313,67,0.621,0.105,0.051,NaN,False,"crispr_invivo_composite, gse108740, gse168465"
4,Tg,5,isomerase,1313,68,0.496,0.108,0.052,NaN,False,"gse245775, toxodb_codon_usage, crispr_glucose_limitation"
5,Tg,6,ligase,1313,92,0.878,0.557,0.07,0.316,False,"proteome_total, oocyst_itraq, gse51780"
6,Tg,7,translocase,1313,75,0.845,0.583,0.057,NaN,False,"unregistered, pride_acetylation, foldseek_struct"
7,Pf,1,oxidoreductase,1220,108,0.745,0.277,0.089,NaN,False,"plasmodb_pf3d7_expression, pf_mrna_dynamics, pf_foldseek_struct"
8,Pf,2,transferase,1220,458,0.601,0.494,0.375,0.402,False,"pf_foldseek_struct, pf_alphafold_confidence, plasmodb_pf3d7_attributes"
9,Pf,3,hydrolase,1220,469,0.789,0.766,0.384,NaN,False,"pf_alphafold_confidence, pf_hsp90_chemoproteome, pf_riboseq"


## 2 · The claims

One row per unannotated gene and class. `called`: yes, at the calibrated threshold that made the class held-out precision. `fills`: called AND the class may fill. Ranked best first within organism.

In [3]:
C = C.sort_values(['organism', 'calibrated'], ascending=[True, False]).reset_index(drop=True)
C.to_parquet(os.path.join(OUT, 'functional_claims.parquet'), index=False)
C.to_parquet(os.path.join(REPO, 'parascape/data/functional_claims.parquet'), index=False)
summary = C.groupby(['organism', 'ec_class', 'name']).agg(genes=('gene_id', 'size'), called=('called', 'sum'), fills=('fills', 'sum')).reset_index()
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t', index=False)
json.dump(dict(claims=len(C), called=int(C.called.sum()), fills=int(C.fills.sum()), classes=len(G)),
          open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
summary

,organism,ec_class,name,genes,called,fills
0,Pf,1,oxidoreductase,4500,46,0
1,Pf,2,transferase,4500,238,0
2,Pf,3,hydrolase,4500,8,0
3,Pf,4,lyase,4500,0,0
4,Pf,5,isomerase,4500,4,0
5,Pf,6,ligase,4500,4,0
6,Pf,7,translocase,4500,6,0
7,Tg,1,oxidoreductase,6827,4,0
8,Tg,2,transferase,6827,146,0
9,Tg,3,hydrolase,6827,27,0
